# 🤗 SetFit

🤗 SetFit is an efficient and prompt-free framework for few-shot fine-tuning of [Sentence Transformers](https://sbert.net/). It achieves high accuracy with little labeled data - for instance, with only 8 labeled examples per class on the Customer Reviews sentiment dataset, 🤗 SetFit is competitive with fine-tuning RoBERTa Large on the full training set of 3k examples!

Compared to other few-shot learning methods, SetFit has several unique features:

* 🗣 **No prompts or verbalizers:** Current techniques for few-shot fine-tuning require handcrafted prompts or verbalizers to convert examples into a format suitable for the underlying language model. SetFit dispenses with prompts altogether by generating rich embeddings directly from text examples.
* 🏎 **Fast to train:** SetFit doesn't require large-scale models like T0, Llama or GPT-4 to achieve high accuracy. As a result, it is typically an order of magnitude (or more) faster to train and run inference with.
* 🌎 **Multilingual support**: SetFit can be used with any [Sentence Transformer](https://huggingface.co/models?library=sentence-transformers&sort=downloads) on the Hub, which means you can classify text in multiple languages by simply fine-tuning a multilingual checkpoint.

In [1]:
%%capture
!pip install datasets setfit torch

### The Two Core Phases of SetFit

SetFit leverages the power of pre-trained Sentence Transformer models but fine-tunes them efficiently for the downstream task in two phases:

#### 1. Siamese Fine-Tuning (Model Fine-Tuning)

This phase's goal is to slightly adjust the Sentence Transformer model so its embeddings are better suited for the target domain (e.g., medical, finance, customer reviews), even with a small dataset.

* **Data Augmentation:** The small set of labeled examples is used to generate **synthetic, labeled pairs of sentences**.
    * **Positive Pairs:** Sentences from the **same class** are paired (e.g., two positive reviews).
    * **Negative Pairs:** Sentences from **different classes** are paired (e.g., a positive review and a negative review).
* **Contrastive Learning:** The Sentence Transformer model is then fine-tuned using these pairs, applying a **contrastive loss function** (a key component of the Siamese network architecture).
    * This forces the model to move the embeddings of positive pairs closer together and push the embeddings of negative pairs farther apart in the vector space.
    * Crucially, this step only requires a **few epochs** and subtly improves the model's ability to distinguish classes based on semantic distance.

#### 2. Head Training (Classifier Training)

Once the Sentence Transformer model is fine-tuned to produce high-quality, task-specific embeddings, this phase trains the final classification component.

* **Embedding Generation:** The now-fine-tuned Sentence Transformer is used to generate a single, static embedding for **every single labeled example**.
* **Simple Classifier:** A fast, small, linear classification head—often a simple **Logistic Regression** model—is trained directly on these embeddings.
* **Speed & Efficiency:** Since the Sentence Transformer model is **frozen** and the final classifier is very lightweight, this step is extremely fast. The heavy computational lifting is over, and the small classifier quickly learns the decision boundary within the optimized vector space.

**Result:** SetFit achieves state-of-the-art results on few-shot classification benchmarks, often **outperforming full fine-tuning** of large models like BERT, while being significantly faster and dramatically reducing the need for extensive labeled data.

### How does Contrastive Learning Work? Anchors, Positives, and Negatives

Contrastive learning operates on triplets or pairs of data points, centered around an "anchor" example:

1.  **The Anchor ($\mathbf{x}_a$):** The reference data point (e.g., a sentence, an image, or a word).
2.  **The Positive ($\mathbf{x}_p$):** A data point that is semantically **similar** to the Anchor.
    * *In NLP:* A paraphrased sentence, a translation, or a sentence from the same class.
    * *In Vision:* An augmented (cropped, rotated) version of the Anchor image.
3.  **The Negative ($\mathbf{x}_n$):** A data point that is semantically **dissimilar** to the Anchor.
    * *In NLP:* A random sentence from a different document or a sentence from a different class.
    * *In Vision:* A randomly selected image from a different part of the dataset.

#### The Goal: Minimizing the Contrastive Loss

The model's job is to learn an embedding function $f(\mathbf{x})$ that maps these data points into a vector space, such that the following relationship holds:

$$\text{Distance}(f(\mathbf{x}_a), f(\mathbf{x}_p)) < \text{Distance}(f(\mathbf{x}_a), f(\mathbf{x}_n))$$

The training uses a specialized **contrastive loss function** (like the Triplet Loss or InfoNCE loss) that enforces this inequality. This loss includes a **margin ($\alpha$ or $\tau$)**, which acts as a penalty threshold, ensuring the distance between the Anchor and the Negative is *at least* that far apart.

$$\text{Loss} = \max\left(0, \text{Distance}(f(\mathbf{x}_a), f(\mathbf{x}_p)) - \text{Distance}(f(\mathbf{x}_a), f(\mathbf{x}_n)) + \alpha\right)$$

By repeatedly minimizing this loss across millions of anchors and their positive/negative counterparts, the model learns a vector space where **semantic similarity is directly proportional to vector proximity**. This is why the resulting embeddings are so effective for tasks like semantic search and classification (as seen in SetFit and SBERT).

In [2]:
from datasets import load_dataset
from setfit import SetFitModel, Trainer, TrainingArguments, sample_dataset
import torch
import os


def load_data(class_to_predict="manipulative"):
    def preprocess_function(examples):
        if class_to_predict == "manipulative":
            return {
                "sentence": examples["content"],
                "label": int(examples["manipulative"]),
            }
        return {
            "sentence": examples["content"],
            "label": int(
                False
                if examples["techniques"] == None
                else class_to_predict in examples["techniques"]
            ),
        }

    data = load_dataset("robinhad/manipulation-detection", split="train")

    # create stratified eval split
    data = data.train_test_split(test_size=0.3, seed=42)
    train_dataset = data["train"]
    eval_dataset = data["test"].train_test_split(test_size=0.5, seed=42)
    test_dataset = eval_dataset["test"]
    eval_dataset = eval_dataset["train"]

    print("Dataset sizes:", len(train_dataset), len(eval_dataset), len(test_dataset))

    train_dataset = train_dataset.map(
        preprocess_function, remove_columns=train_dataset.column_names
    ) 
    eval_dataset = eval_dataset.map(
        preprocess_function, remove_columns=eval_dataset.column_names
    )
    test_dataset = test_dataset.map(
        preprocess_function, remove_columns=test_dataset.column_names
    )

    return train_dataset, eval_dataset, test_dataset

/home/robinhad/Projects/ucu-nlp-course/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class_to_predict = "manipulative"
print(f"Predicting class: {class_to_predict}")

train_dataset, eval_dataset, test_dataset = load_data(class_to_predict)


model = SetFitModel.from_pretrained(
    "intfloat/multilingual-e5-large-instruct", # Alibaba-NLP/gte-multilingual-base",
    labels=["normal", "manipulative"],
    trust_remote_code=True,
)


args = TrainingArguments(
    batch_size=(6, 2),
    # num_epochs=(1, 4),
    # eval_strategy="epoch",
    save_strategy="epoch",
    # load_best_model_at_end=True,
    max_steps=100,
    body_learning_rate=(0.0001269119351966899, 6.333356345219366e-05),
    head_learning_rate=2.714153430200578e-05,
    output_dir=class_to_predict,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    metric="f1",
    column_mapping={
        "sentence": "text",
        "label": "label",
    },  # Map dataset columns to text/label expected by trainer
)

# Train and evaluate
trainer.train()

Predicting class: manipulative
Dataset sizes: 2675 573 574


/home/robinhad/Projects/ucu-nlp-course/env/lib/python3.12/site-packages/datasets/utils/_dill.py:385: DeprecationWarning: co_lnotab is deprecated, use co_lines instead.
  obj.co_lnotab,  # for < python 3.10 [not counted in args]
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset
Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.
/home/robinhad/Projects/ucu-nlp-course/en

Step,Training Loss
1,0.130600
50,0.289000
100,0.263100


In [4]:

# check if inside main process
# if torch.distributed.get_rank() == 0:
metrics = trainer.evaluate(eval_dataset)
trainer.model.save_pretrained(class_to_predict)
#import json

#with open(f"{class_to_predict}_metrics.json", "w") as f:
#    json.dump(metrics, f)
print(metrics)

Applying column mapping to the evaluation dataset
***** Running evaluation *****


{'f1': 0.7924130663856691}


In [5]:
# Save model
# trainer.save_model("model")

# {'accuracy': 0.792413066385669}
# %%
# Push model to the Hub
# model_link = "<your-username>/<model-name>"

# trainer.push_to_hub(model_link)

# Download from Hub
# model = SetFitModel.from_pretrained(model_link)
# Run inference
# preds = model.predict(
#    ["i loved the spiderman movie!", "pineapple on pizza is the worst 🤮"]
# )
# print(preds)